In [ ]:
#| default_exp bench

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, sys, os, ctypes, platform, subprocess
from fastcore.all import AttrDict

## Measuring a process

Three numbers per process. `rss` is resident memory, which counts the table's mapped pages once they are touched. `private` is what the
process holds that no other process can share and the kernel cannot drop without writing it out: `phys_footprint` on macOS (Activity
Monitor's Memory column), `RssAnon` on Linux. A mapped, clean table counts in the first and not the second, so `private` is the number
that grows with each worker. `peak` is the most `private` has been over the process's life on macOS (`lifetime_max_phys_footprint`),
and the peak resident size on Linux (`VmHWM`), which counts mapped pages too.

In [ ]:
#| export
def footprint():
    "This process's resident, private and peak memory in MB."
    if platform.system() == 'Darwin':
        buf = (ctypes.c_uint64 * 64)()   # rusage_info_v4 as uint64s: a 16-byte uuid, then resident_size at 8, phys_footprint 9, lifetime_max_phys_footprint 30
        ctypes.CDLL(None).proc_pid_rusage(ctypes.c_int(os.getpid()), 4, buf)
        return AttrDict(rss=buf[8] / 2**20, private=buf[9] / 2**20, peak=buf[30] / 2**20)
    kv = dict(l.split(':', 1) for l in open('/proc/self/status'))
    mb = lambda k: int(kv[k].split()[0]) / 1024
    return AttrDict(rss=mb('VmRSS'), private=mb('RssAnon'), peak=mb('VmHWM'))

In [ ]:
f = footprint(); assert 0 < f.private <= f.peak and f.rss > 0, f

`measure` loads a model in a fresh interpreter, one way or the other, and encodes `corpus(n)` with it. `check` runs `same_as_model2vec`
in a fresh interpreter too and adds what the model is: tokenizer type, vocabulary, dimensions, dtype, `normalize`. A process each keeps
one measurement from inheriting another's heap, and keeps a dozen large models from piling up in the notebook.

In [ ]:
#| export
_PROBE = '''
import json, time
from laghu.bench import footprint
from laghu.core import corpus, resolve
t = time.time()
if {how!r} == 'model2vec':
    from model2vec import StaticModel
    m = StaticModel.from_pretrained(str(resolve({path!r})), force_download=False)
else:
    from laghu import load
    m = load({path!r}, fallback=False)
tl = time.time() - t; f1 = footprint()
txts = corpus({n}); t = time.time(); m.encode(txts); te = time.time() - t; f2 = footprint()
print(json.dumps(dict(how={how!r}, load_s=tl, encode_s=te, texts=len(txts), private_load=f1.private, private=f2.private, rss=f2.rss, peak=f2.peak)))
'''

_CHECK = '''
import json
from laghu import load, same_as_model2vec
from laghu.core import corpus, model_files, resolve, find_vocab
m = load({path!r}, fallback=False); f = model_files(m.folder)
kind = json.loads(f.tok.read_text())['model'].get('type') or 'Unigram'
r = same_as_model2vec({path!r}, corpus({n}))
print(json.dumps(dict(r, kind=kind, vocab=len(m.embedding), dim=m.dim, dtype=m.embedding_dtype, normalize=bool(m.normalize),
    quantized=m.token_mapping is not None, weighted=m.weights is not None)))
'''

def _run(code):
    r = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True)
    if r.returncode: raise RuntimeError(r.stderr[-3000:])
    return AttrDict(json.loads(r.stdout.strip().splitlines()[-1]))

def measure(path, how='laghu', n=2000):
    "Memory (MB) and time (s) of loading `path` with `how` ('laghu' or 'model2vec') and encoding `corpus(n)`, in a new process."
    return _run(_PROBE.format(path=str(path), how=how, n=n))

def check(path, n=3000):
    "`same_as_model2vec(path, corpus(n))` and the model's shape, in a new process."
    return _run(_CHECK.format(path=str(path), n=n))

def row(path, n_check=3000, n_speed=2000):
    "One model's line: what it is, whether it matches, and both ways' memory and speed."
    r = check(path, n_check); r.model = str(path)
    r.m2v, r.lg = measure(path, 'model2vec', n_speed), measure(path, 'laghu', n_speed)
    return r

In [ ]:
#| export
def same(r): return all(r[k] for k in ('ids', 'dtype', 'f32', 'f16', 'exact', 'seq', 'median', 'unk', 'tokens'))

def table(rows):
    "Rows of `row` as a markdown table."
    hd = ('| model | tokenizer | laghu tokenizes with | vocab × dims | dtype | norm | identical | private MB, model2vec → laghu | peak MB, model2vec → laghu | texts/s, model2vec → laghu |\n'
          '|---|---|---|---|---|---|---|---|---|---|\n')
    def ln(r):
        extra = ' +map' * r.quantized + ' +weights' * r.weighted
        return (f"| {r.model.split('/')[-1]} | {r.kind} | {'Python Viterbi' if r.lean else 'tokenizers'} | {r.vocab:,} × {r.dim} | {r.dtype}{extra} | "
                f"{'on' if r.normalize else 'off'} | {'yes' if same(r) else 'NO'} | {r.m2v.private:,.0f} → {r.lg.private:,.0f} | {r.m2v.peak:,.0f} → {r.lg.peak:,.0f} | "
                f"{r.m2v.texts / r.m2v.encode_s:,.0f} → {r.lg.texts / r.lg.encode_s:,.0f} |")
    return hd + '\n'.join(map(ln, rows))

## Results

On an M3 Max, macOS 26, Python 3.13, model2vec 0.9.0, tokenizers 0.23.2. Equivalence is checked on `corpus(3000)`, 4,726 texts (ids, the
vectors cast to float32 and to float16, the exact dtype and bits, `encode_as_sequence` on 200 of them, `median_token_length`, the unknown
id, the token list); speed is `encode` over `corpus(2000)`, 3,226 texts. `private` is measured after encoding. These runs download about
5 GB of models, so the cell is flagged `slow`: `nbdev-test --flags slow` runs it. Both need model2vec (`laghu[fallback]`).

`tokenizers` encodes a batch on every core; the Python Viterbi uses one, so on a Unigram model laghu encodes at a third to a half of
model2vec's rate on this machine. Everything else encodes at the same rate or faster, since the work is the same and the table is not copied.

In [ ]:
#| slow
from IPython.display import Markdown
from laghu.core import try_resolve
import importlib.util
M2V = importlib.util.find_spec('model2vec') is not None   # without it there is nothing to compare with: skip
MODELS = [
    'minishlab/potion-base-2M', 'minishlab/potion-base-4M', 'minishlab/potion-base-8M', 'minishlab/potion-base-32M', 'minishlab/potion-retrieval-32M',
    'minishlab/potion-science-8M', 'minishlab/potion-code-16M', 'minishlab/potion-code-16M-v2', 'minishlab/potion-multilingual-128M',
    'minishlab/M2V_base_output', 'minishlab/M2V_multilingual_output', 'minishlab/M2V_base_glove', 'minishlab/M2V_base_glove_subword',
    'CISCai/jina-embeddings-v3-separation-distilled', 'nexus666/multilingual-e5-large-m2v', 'mikeee/m2v-gemma-embedding-300m',
    'Tokenade/potion-multilingual-128M-i8-tokenade', 'tss-deposium/m2v-qwen3-embedding-0.6b-1024d', 'OpenVoiceOS/ovos-m2v-intents-es-ES-bne-v5',
    'JanSchachtschabel/m2v-gte-256-edu', 'juampahc/bge-m3-m2v-256', 'blobbybob/potion-mxbai-micro', 'stephantulkens/NIFE-mxbai-embed-large-v1_model2vec',
    'OpenVoiceOS/ovos-m2v-intents-pt-PT-albertina-v5', 'giangndm/F2LLM-v2-330M-model2vec', 'amgix/static-retrieval-multilingual-69m-v1']
rows, failed = [], {}
for p in MODELS * M2V:
    if not try_resolve(p): continue
    try: rows.append(row(p))
    except Exception as e: failed[p] = repr(e)[-300:]
print(failed)
Markdown(table(rows))

In [ ]:
#| slow
assert not M2V or rows and not failed and all(same(r) for r in rows), [r.model for r in rows if not same(r)]